In [2]:
import pandas as pd 
import os 
import requests
from openai import OpenAI
from dotenv import load_dotenv
import urllib3
import requests
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By 
from datetime import datetime as dt
import undetected_chromedriver as uc
from twocaptcha import TwoCaptcha
import json
load_dotenv()
import sys
sys.path.append(os.path.abspath(".."))
from LLM_Agent.llm_template import LLMAgent
from selenium.common.exceptions import TimeoutException
from playwright.async_api import async_playwright
import pdfplumber
import io
from transformers import AutoTokenizer
from huggingface_hub import login
import logging
import sys
from time import sleep 
import time
from urllib.parse import urljoin
from pdf2image import convert_from_bytes
import pytesseract
import asyncio
import time
import io
import pdfplumber
import httpx
from playwright_stealth import stealth_async
import random
from urllib.parse import urljoin, urlparse, urlunparse, parse_qs, urlencode
import random


In [3]:
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 13_2) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:123.0) Gecko/20100101 Firefox/123.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 13_1) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.0 Safari/605.1.15"
]

In [5]:
login(os.getenv("hugging_face_token"))
logging.getLogger("pdfminer").setLevel(logging.ERROR)
sys.stderr = open(os.devnull, 'w')

In [6]:
get_pdf_href_prompt = f"""


You are an assistant that analyzes the HTML of academic paper webpages to extract the direct PDF download link.

Given a the url to a scientific paper, perform the following:

1. Follow any redirects to land on the final page hosting the full paper.
2. Analyze the HTML and look for anchor (`<a>`) tags that link to PDF documents.
   - These usually contain "pdf" in the `href`
   - May use a button or text like "Download PDF", "View PDF", etc.
3. Return only the absolute URL of the PDF 

Example URL:
https://doi.org/10.1111/bph.15505

Expected Output Format:
https://bpspubs.onlinelibrary.wiley.com/doi/epdf/10.1111/bph.15505

"""




clean_text_prompt = f"""
    The following text is extracted from a research paper. Your task is to:
    - clean it up with minimal 
    - Keep only the **main body content** starting from the abstract up to the conclusion/discussion.
    - Remove footnotes, references, figure captions, and legal disclaimers.
    - Ensure proper paragraph structure and readability.
    - Preserve the logical order of content.
    - Do not include any commentary

    Here is the raw extracted text:
    

    """



system_prompt_1 = (
    "You are an AI agent helping a researcher extract and clean scientific documents, "
    "extract metadata, and fetch PDF links. Respond with the ouput result always"
)

In [7]:
api_key = os.getenv("OPENAI_API_KEY")
model_name = 'Llama-3-8B-Instruct-exl2'
agent = LLMAgent(model_name)
# agent.unload_and_load_model(model_name)

In [ ]:
def chunk_text_by_tokens(text, max_tokens=8000):
    if not isinstance(text, str):
        raise ValueError("`text` must be a string. Got: {}".format(type(text)))
        
    tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")
    input_ids = tokenizer.encode(text, add_special_tokens=False)
    
    chunks = []
    for i in range(0, len(input_ids), max_tokens):
        chunk_ids = input_ids[i:i + max_tokens]
        chunk = tokenizer.decode(chunk_ids)
        chunks.append(chunk)
    
    return chunks


def chunk_text_by_char_limit(text, limit=8000):
    return [text[i:i+limit] for i in range(0, len(text), limit)]

def extract_text_with_ocr(pdf_bytes):
    images = convert_from_bytes(pdf_bytes)
    full_text = []
    for i, img in enumerate(images):
        text = pytesseract.image_to_string(img)
        if text:
            full_text.append(text)
    return "\n".join(full_text).strip() if full_text else None

def extract_text_with_ocr(pdf_bytes):
    images = convert_from_bytes(pdf_bytes)
    full_text = []
    for i, img in enumerate(images):
        text = pytesseract.image_to_string(img)
        if text:
            full_text.append(text)
    return "\n".join(full_text).strip() if full_text else None



def extract_pdf(pdf_data): 
     
    extracted_text = []

    try:
        with pdfplumber.open(io.BytesIO(pdf_data)) as pdf:
                for pdf_page in pdf.pages:
                    page_text = pdf_page.extract_text()
                    if page_text:
                        extracted_text.append(page_text)
    except Exception as e:
        print(f"pdfplumber failed: {e}")

    if not extracted_text:
        print("Falling back to OCR...")
        ocr_text = extract_text_with_ocr(pdf_data)
        return ocr_text

    return "\n".join(extracted_text).strip() if extracted_text else None


async def extract_text_from_pdf_via_browser(landing_url: str):
    try:
        login_url = 'https://login.ezproxy.lib.ucalgary.ca/login'
        ezproxy_prefix = 'https://ezproxy.lib.ucalgary.ca/login?url='
        proxied_url = ezproxy_prefix + landing_url
        random_ua = random.choice(USER_AGENTS)

        username = os.getenv('uni_username')
        password = os.getenv('uni_password')
        if not username or not password:
            raise ValueError("University EZProxy credentials not found in environment variables")

        async with async_playwright() as p:
            browser = await p.chromium.launch(
                headless=False,
                args=[
                    "--disable-blink-features=AutomationControlled",
                    "--disable-web-security",
                    "--no-sandbox",
                    "--disable-gpu"
                ]
            )
            context = await browser.new_context(
                user_agent=random_ua,
                locale='en-US',
                timezone_id='America/New_York'
            )
            await context.add_init_script("""
                Object.defineProperty(navigator, 'webdriver', { get: () => undefined });
                Object.defineProperty(navigator, 'languages', { get: () => ['en-US', 'en'] });
                Object.defineProperty(navigator, 'plugins', { get: () => [1, 2, 3] });
            """)
            page = await context.new_page()

            try:
                from playwright_stealth import stealth_async
                await stealth_async(page)
            except ImportError:
                print("playwright_stealth not installed; continuing without stealth")

            print(f"Navigating to EZProxy URL: {proxied_url}")
            await page.goto(proxied_url, wait_until="load")
            await page.wait_for_timeout(30000)

            current_url = page.url
            if current_url.lower().endswith(".pdf") or "pdf" in current_url.lower():
                print(f"Already on a PDF page: {current_url}")
                response = await context.request.get(current_url)
                if response.status == 200 and "pdf" in response.headers.get("content-type", "").lower():
                    print("PDF detected via URL.")
                    content = await response.body()
                    text = extract_pdf(content)
                    if text: return text

            try:
                pdf_href = await page.get_by_role("link", name="PDF").get_attribute("href")
                if pdf_href:
                    resolved_link = urljoin(page.url, pdf_href)
                    response = await context.request.get(resolved_link)
                    if response.status == 200 and "pdf" in response.headers.get("content-type", "").lower():
                        print("PDF extracted from PDF button.")
                        content = await response.body()
                        text = extract_pdf(content)
                        if text: return text
            except:
                pass

            print("🔍 Searching nested button group structure for download link...")
            try:
                download_group = page.locator("div.grouped.right")
                buttons = await download_group.locator("a.navbar-download.btn.btn--cta.roundedColored").all()
                for button in buttons:
                    href = await button.get_attribute("href")
                    if href and "pdf" in href:
                        pdf_link = urljoin(page.url, href)
                        print(f"📄 PDF found under nested class structure: {pdf_link}")
                        response = await context.request.get(pdf_link)
                        if response.status == 200 and "pdf" in response.headers.get("content-type", "").lower():
                            content = await response.body()
                            text = extract_pdf(content)
                            if text: return text
            except Exception as e:
                print(f"⚠️ Failed extracting nested class-based PDF link: {e}")

            print("Attempting fallback reconstruction from epdf link...")
            epdf_url = landing_url
            if "/epdf/" in landing_url:
                try:
                    parsed = urlparse(epdf_url)
                    parts = parsed.path.split("/epdf/")
                    if len(parts) == 2:
                        prefix, suffix = parts[0], parts[1]
                        pdf_variants = ["pdf", "pdfdirect", "pdfdownload"]
                        cookies = await context.cookies()
                        await browser.close()

                        cookie_dict = {c["name"]: c["value"] for c in cookies}
                        headers = {
                            "User-Agent": random_ua,
                            "Accept": "application/pdf",
                            "Cookie": "; ".join(f"{k}={v}" for k, v in cookie_dict.items()),
                            "Referer": landing_url
                        }

                        for variant in pdf_variants:
                            reconstructed = f"{parsed.scheme}://{parsed.netloc}{prefix}/{variant}/{suffix}?download=False"
                            print(f"Trying reconstructed URL: {reconstructed}")
                            async with httpx.AsyncClient(follow_redirects=True) as client:
                                resp = await client.get(reconstructed, headers=headers)
                                print(f"ℹ️ Status: {resp.status_code}, Content-Type: {resp.headers.get('Content-Type')}")
                                if resp.status_code == 200:
                                    content_type = resp.headers.get("Content-Type", "").lower()
                                    if "pdf" in content_type:
                                        print("✅ PDF detected via reconstructed URL.")
                                        text = extract_pdf(resp.content)
                                        if text: return text
                except Exception as e:
                    print(f"Error during fallback reconstruction: {e}")

            cookies = await context.cookies()
            await browser.close()

        cookie_dict = {cookie["name"]: cookie["value"] for cookie in cookies}
        headers = {
            "Cookie": "; ".join(f"{k}={v}" for k, v in cookie_dict.items()),
            "User-Agent": random_ua
        }

        async with httpx.AsyncClient(follow_redirects=True, timeout=30.0) as client:
            response = await client.get(landing_url, headers=headers)
            if response.status_code == 200 and "pdf" in response.headers.get("content-type", "").lower():
                print("PDF fetched directly from landing URL.")
                return extract_pdf(response.content)

        print("All extraction methods failed.")
        return None

    except Exception as e:
        print(f"Error during extraction: {e}")
        return None

    
async def get_biorxiv_pdf_link(article_url: str) -> str:
    try:
        random_ua = random.choice(USER_AGENTS)

        async with async_playwright() as p:
            browser = await p.chromium.launch(
                headless=False,
                args=[
                    "--disable-blink-features=AutomationControlled",
                    "--disable-web-security",
                    "--no-sandbox",
                    "--disable-gpu"
                ]
            )

            context = await browser.new_context(
                user_agent=random_ua,
                viewport={"width": 1280, "height": 720},
                locale='en-US',
                timezone_id='America/New_York',
                geolocation={"longitude": -114.0719, "latitude": 51.0447},
                permissions=["geolocation"]
            )

            await context.add_init_script("""
                Object.defineProperty(navigator, 'webdriver', { get: () => undefined });
                Object.defineProperty(navigator, 'languages', { get: () => ['en-US', 'en'] });
                Object.defineProperty(navigator, 'plugins', { get: () => [1, 2, 3] });
            """)

            page = await context.new_page()

            try:
                from playwright_stealth import stealth_async
                await stealth_async(page)
            except ImportError:
                print("playwright-stealth not installed. Proceeding without stealth mode.")

            print(f"Navigating to: {article_url}")
            await page.goto(article_url, wait_until="networkidle")
            await page.wait_for_timeout(30000)

            current_url = page.url
            if current_url.lower().endswith(".pdf") or "pdf" in current_url.lower():
                print(f"Already on PDF page: {current_url}")
                pdf_link = current_url
            else:
                pdf_href = None
                try:
                    await page.wait_for_selector("a.article-dl-pdf-link", timeout=30000)
                    pdf_href = await page.locator("a.article-dl-pdf-link").get_attribute("href")
                except Exception as e:
                    print(f"Could not find PDF link via class selector: {e}")

                if not pdf_href:
                    print("No PDF link found.")
                    await browser.close()
                    return None

                pdf_link = urljoin(page.url, pdf_href)
                print(f"Final PDF URL: {pdf_link}")

            cookies = await context.cookies()
            await browser.close()

            cookie_dict = {cookie["name"]: cookie["value"] for cookie in cookies}
            headers = {
                "Cookie": "; ".join(f"{k}={v}" for k, v in cookie_dict.items()),
                "User-Agent": random_ua,
                "Accept": "text/html,application/pdf;q=0.9,*/*;q=0.8",
                "Accept-Language": "en-US,en;q=0.9",
                "Referer": article_url,
                "Connection": "keep-alive",
                "DNT": "1"
            }

            async with httpx.AsyncClient(follow_redirects=True) as client:
                response = await client.get(pdf_link, headers=headers)
                response.raise_for_status()
                content_type = response.headers.get("Content-Type", "")
                if "pdf" not in content_type.lower():
                    print(f"Unexpected Content-Type: {content_type}")
                    return None

                pdf_data = response.content
                extracted_text = extract_pdf(pdf_data)
                return extracted_text

    except Exception as e:
        print(f"Error: {e}")
        return None



        

In [9]:
http = urllib3.PoolManager()

biorxiv_url_endpoint = 'https://api.biorxiv.org/pubs/biorxiv/2022-01-01/2025-01-01/1'
pre_print_response = http.request("GET", biorxiv_url_endpoint)
pre_print_decoded = pre_print_response.data.decode("utf-8")
pre_print_parsed = json.loads(pre_print_decoded)
paper_metadeta = pre_print_parsed['collection']

In [14]:
paper_metadeta_approved_list = []
paper_metadeta_unextracted_list = []

count = 0
for paper in paper_metadeta:
    doi_types = {
        'preprint_doi': f"https://www.biorxiv.org/content/{paper['preprint_doi']}",
        'published_doi': f"https://doi.org/{paper['published_doi']}"
    }

    research_text_bucket = []
    paper_dict = {
        "preprint_doi": "",
        "published_doi": "",
        "published_journal": "",
        "preprint_title": "",
        "preprint_authors": "",
        "preprint_category": "",
        "preprint_date": "",
        "published_date": "",
        "preprint_author_corresponding": "", 
        "preprint_author_corresponding_institution": "",
        "preprint_paper": "",
        "published_paper": "",
    }

    for paper_key, paper_url in doi_types.items(): 
        try:
            print(f'Extracting {paper_key}')
            if paper_key == "preprint_doi":
                relevant_text = await get_biorxiv_pdf_link(paper_url)
            else:
                pdf_href = agent.one_turn(
                    system_prompt="""
                        You are an assistant that analyzes the HTML of academic paper webpages to extract the direct PDF download link.

                        Given the URL to a scientific paper, perform the following:
                        1. Follow any redirects to land on the final page hosting the full paper.
                        2. Analyze the HTML and look for anchor (`<a>`) tags that link to PDF documents.
                           - These usually contain "pdf" in the `href`
                           - May use a button or text like "Download PDF", "View PDF", etc.
                        3. Return only the absolute URL to the actual `.pdf` file — it must be a direct link to a downloadable PDF (not an HTML viewer or embedded viewer).
                        4. If a PDF link contains `download=true`, return the **same link with `download=false` instead**.
                        5. Do not include any commentary or analysis

                        Examples:
                        Input: https://doi.org/10.1111/bph.15505
                        Output: https://bpspubs.onlinelibrary.wiley.com/doi/pdfdirect/10.1111/bph.15505?download=true
                        Input: https://www.biorxiv.org/content/10.1101/2020.06.02.130062v2
                        Output: https://www.biorxiv.org/content/10.1101/2020.06.02.130062v2.full.pdf
                        Input: https://bpspubs.onlinelibrary.wiley.com/doi/pdfdirect/10.1111/bph.15505?download=true
                        Output: https://bpspubs.onlinelibrary.wiley.com/doi/pdfdirect/10.1111/bph.15505?download=False
                    """,
                    user_prompt=paper_url
                )
                relevant_text = await extract_text_from_pdf_via_browser(pdf_href)

            if relevant_text is None:
                print(f"Could not Get {paper_key} Paper .. moving to next paper")
                continue

            print(f"Extracted {paper_key}")
            chunk_text_storage = []
            chunks = chunk_text_by_char_limit(relevant_text)

            for chunk in chunks:
                cleaned_chunk = agent.one_turn(
                    system_prompt="""
                        The following text is extracted from a research paper. Your task is to:
                        - Clean it up
                        - Keep only the **main body content**
                        - Remove footnotes, references, figure captions, and legal disclaimers.
                        - Ensure proper paragraph structure and readability.
                        - Preserve the logical order of content.
                        - Do not include any commentary or analysis
                    """,
                    user_prompt=chunk
                )
                chunk_text_storage.append(cleaned_chunk)

            print(f"Cleaned {paper_key}")
            cleaned_text = " ".join(chunk_text_storage)
            research_text_bucket.append(cleaned_text)

        except TimeoutException:
            print('Time out while loading')
            continue
        except Exception as e:
            print(e)
            continue

    if len(research_text_bucket) == 2:
        paper_dict.update({
            "preprint_doi": paper["preprint_doi"],
            "published_doi": paper["published_doi"],
            "published_journal": paper["published_journal"],
            "preprint_title": paper["preprint_title"],
            "preprint_authors": paper["preprint_authors"],
            "preprint_category": paper["preprint_category"],
            "preprint_date": paper["preprint_date"],
            "published_date": paper["published_date"],
            "preprint_author_corresponding": paper["preprint_author_corresponding"],
            "preprint_author_corresponding_institution": paper["preprint_author_corresponding_institution"],
            "preprint_paper": research_text_bucket[0],
            "published_paper": research_text_bucket[1]
        })

        paper_metadeta_approved_list.append(paper_dict)
        count += 1
        print(f"Papers Extracted: {count}")
    else:
        paper_dict.update({
            "preprint_doi": paper["preprint_doi"],
            "published_doi": paper["published_doi"],
            "published_journal": paper["published_journal"],
            "preprint_title": paper["preprint_title"],
            "preprint_authors": paper["preprint_authors"],
            "preprint_category": paper["preprint_category"],
            "preprint_date": paper["preprint_date"],
            "published_date": paper["published_date"],
            "preprint_author_corresponding": paper["preprint_author_corresponding"],
            "preprint_author_corresponding_institution": paper["preprint_author_corresponding_institution"],
            "preprint_paper": "N/A",
            "published_paper": 'N/A'
        })
        paper_metadeta_unextracted_list.append(paper_dict)

        print("Could not Get Published and PrePrint Paper Pairs, stored paper info and moving on...")
        continue

    if count >= 5:
        break



Extracting preprint_doi
Navigating to: https://www.biorxiv.org/content/10.1101/2020.12.16.423137
Final PDF URL: https://www.biorxiv.org/content/10.1101/2020.12.16.423137v2.full.pdf
Error: 
Could not Get preprint_doi Paper .. moving to next paper
Extracting published_doi
Navigating to EZProxy URL: https://ezproxy.lib.ucalgary.ca/login?url=https://www.sciencedirect.com/science/article/pii/S2211124721009285/pmc_articles/PDF/10.1016/j.celrep.2021.110124.pdf
Already on a PDF page: https://www-sciencedirect-com.ezproxy.lib.ucalgary.ca/science/article/pii/S2211124721009285/pmc_articles/PDF/10.1016/j.celrep.2021.110124.pdf
🔍 Searching nested button group structure for download link...
📦 Attempting fallback reconstruction from epdf link...
❌ All extraction methods failed.
Could not Get published_doi Paper .. moving to next paper
Could not Get Published and PrePrint Paper Pairs, stored paper info and moving on...
Extracting preprint_doi
Navigating to: https://www.biorxiv.org/content/10.1101/2020

In [15]:
paper_metadeta_approved_list

[{'preprint_doi': '10.1101/2021.08.26.457858',
  'published_doi': '10.1038/s41380-021-01402-9',
  'published_journal': 'Molecular Psychiatry',
  'preprint_title': 'A thalamo-centric neural signature for restructuring negative self-beliefs',
  'preprint_authors': 'Steward, T.; Kung, P.-H.; Davey, C. G.; Moffat, B. A.; Glarin, R. K.; Jamieson, A. J.; Felmingham, K. L.; Harrison, B. J.',
  'preprint_category': 'neuroscience',
  'preprint_date': '2021-08-28',
  'published_date': '2022-01-01',
  'preprint_author_corresponding': 'Trevor  Steward',
  'preprint_author_corresponding_institution': 'University of Melbourne',
  'preprint_paper': "A Thalamo-Centric Neural Signature for Restructuring Negative Self-Beliefs\n\nBeliefs that are negatively biased, inaccurate, and rigid play a key role in the etiology and maintenance of psychopathology. Cognitive models of mood disorders posit that maladaptive self-beliefs - for example, believing that one is inherently flawed or unlovable - are central 